# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Use the `@id` fields for precise referencing.

Let's enumerate the record sets (`@id`), and for each record set, print its fields and their `@id`s.

In [ ]:
print("Available record sets in dataset (by @id):\n")
record_sets = []
for rs in metadata.record_sets:
    print(f"  RecordSet @id: {rs.id} | name: {getattr(rs, 'name', '')}")
    record_sets.append(rs.id)
    print("    Fields in this record set:")
    for field in rs.fields:
        print(f"      Field @id: {field.id} | name: {getattr(field, 'name', '')} | dataType: {getattr(field, 'data_type', '')}")
    print("\n")
if not record_sets:
    print("No record sets found in metadata! Please check the schema or contact the dataset provider.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

_Note: Use the record set and field `@id`s listed above._

In [ ]:
# Prepare to extract from all available record sets
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found for record set: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping data. All references are made using `@id` fields.

> _Let's assume the "main" record set is the first one, and select the first numeric field available for demonstration._

In [ ]:
# Identify the first record set and select a numeric field by its @id
main_record_set_id = record_sets[0]
main_df = dataframes[main_record_set_id]

# Find a numeric field in the main record set
main_record_set = next(rs for rs in metadata.record_sets if rs.id == main_record_set_id)
numeric_field_id = None
for field in main_record_set.fields:
    if field.data_type in ('schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number', 'schema:Number'):
        numeric_field_id = field.id
        print(f"Selected numeric field: {numeric_field_id} ({field.name})")
        break
if not numeric_field_id:
    raise RuntimeError('No numeric field found for EDA.')

# Apply filtering and normalization
col = numeric_field_id
# Some values might be missing or not numeric; coerce and drop
main_df[col] = pd.to_numeric(main_df[col], errors='coerce')
filtered_df = main_df[main_df[col] > 10].copy()
print(f"Filtered records with {col} > 10: {len(filtered_df)} rows")
print(filtered_df.head())

filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
print(f"\nNormalized {col} for filtered records:")
print(filtered_df[[col, f"{col}_normalized"]].head())

# Try grouping by a categorical field
group_field_id = None
for field in main_record_set.fields:
    if field.data_type in ('schema:Text', 'Text', 'String', 'schema:String', 'schema:Category') and field.id != col:
        group_field_id = field.id
        print(f"Will group by: {group_field_id} ({field.name})")
        break
# Check field exists
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[col].mean().reset_index()
    print(f"\nMean of {col} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field, and if a group field was selected, plot group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[col].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {col}")
plt.xlabel(col)
plt.ylabel('Frequency')
plt.show()

# If group_field_id is available, plot group means
if 'grouped_df' in locals():
    plt.figure(figsize=(10,4))
    sns.barplot(x=group_field_id, y=col, data=grouped_df)
    plt.title(f"Mean {col} by {group_field_id}")
    plt.xticks(rotation=60)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library. We examined available record sets and fields, loaded main tables into DataFrames, performed numeric filtering and normalization, grouped results by categorical field, and visualized outcome distributions.

_Feel free to further tailor the analysis or EDA steps to your specific research questions or analysis needs!_
